# Method 5: RePaint (unsupervised diffusion)
Unconditional DDPM trained only on clean AFHQ (no masks). RePaint inpaints zero-shot: at every step only the hole is generated, the known region is replaced by the correctly noised real pixels, with resampling jumps. Backbone: U-Net baseline, optionally U-ViT. Slow sampling: evaluate on a subset.

**Code lives in** `src/methods/repaint.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !{sys.executable} scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Diffusion model (use a pretrained AFHQ checkpoint if one exists, else train at 128x128 or 64x64)

In [ ]:
from src.methods.repaint import RePaint
rp = RePaint(model=None, T=250, jump_length=10, resamples=10, device=dev)

## 3. Evaluate (K=5 fills, subset)

In [ ]:
from src.eval import evaluate, save_result
for mask in ['square', 'one_sided', 'centre']:
    res = evaluate(rp, mask, n_images=100, device=dev, save_dir=f'outputs/samples/{rp.name}_{mask}')
    save_result(rp.name, mask, res)
    print(mask, res)

## 4. (Optional) noise-distribution study (professor's feedback)

In [ ]:
# replace Gaussian noise in sampling with uniform / Laplace of matched variance and measure degradation